# Carbon-Aware Recommender System — Full Pipeline

This notebook runs the complete pipeline on Google Colab (with GPU):

| Step | Script | Description |
|------|--------|-------------|
| 1 | `01_train_recommender.py` | RecBole → top-K candidates with `relevance_score` |
| 2 | `02_train_deepfm.py` | DeepFM → `engagement_score` per candidate |
| 3 | `03_rerank.py` | Carbon-aware re-ranking: `s = (1−λ)·engagement − λ·carbon` |
| 4 | `04_evaluate.py` | Pareto frontier, summary tables, plots |

**Runtime**: Select **GPU** under *Runtime → Change runtime type* for faster training.

## 0. Setup

In [ ]:
# ── Clone the repository ──────────────────────────────────────────
import os

REPO_URL = "https://github.com/andersvestrum/carbon-aware-recsys.git"
BRANCH = "2-mapping-carbon---amazon"
REPO_DIR = "/content/carbon-aware-recsys"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin {BRANCH}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

From github.com:andersvestrum/carbon-aware-recsys
 * branch            2-mapping-carbon---amazon -> FETCH_HEAD
Already up to date.
Working directory: /Users/jorgenbergh/Documents/carbon-aware-recsys


### ⚠️ Data Setup

The `data/` folder is gitignored (too large for GitHub), so you need to upload it.

**Option A — Google Drive (recommended):**
1. Upload `data/interim/` to your Google Drive (e.g. `My Drive/carbon-aware-recsys/data/interim/`)
2. Run the cell below to mount Drive and copy the data

**Option B — Direct upload:**
Skip the cell below and manually upload `data/interim/` via the Colab file browser (folder icon on the left).

In [2]:
# ── Mount Google Drive & copy data ────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Configure your Drive path here ───────────────────────────────
# Point this to wherever you uploaded the project data on Drive.
DRIVE_DATA = "/content/drive/MyDrive/carbon-aware-recsys/data"

import shutil
from pathlib import Path

for folder in ["interim", "raw", "processed"]:
    src = Path(DRIVE_DATA) / folder
    dst = Path(REPO_DIR) / "data" / folder
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(str(src), str(dst), dirs_exist_ok=True)  # Robust: allow existing dirs
        print(f"  ✓ Copied {folder}/ ({sum(1 for _ in dst.rglob('*') if _.is_file())} files)")
    else:
        print(f"  ⚠️ {src} not found on Drive — skipping")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# ── Install dependencies ──────────────────────────────────────────
!pip install -q torch recbole deepctr-torch scikit-learn \
    pandas numpy matplotlib seaborn pyyaml joblib tqdm pyarrow \
    omegaconf colorlog kmeans_pytorch

In [ ]:
# ── Monkey-patch for RecBole LightGCN scipy compatibility ──────────────────
import scipy.sparse as sp


def _patch_lightgcn_scipy():
    """
    RecBole's LightGCN uses dok_matrix._update() which doesn't exist in scipy >= 1.13.
    This patches the get_norm_adj_mat method to use standard scipy operations.
    """
    try:
        from recbole.model.general_recommender import LightGCN

        original_get_norm_adj_mat = LightGCN.get_norm_adj_mat

        def patched_get_norm_adj_mat(self):
            """Patched version that works with newer scipy."""
            row = self.edge_index[0]
            col = self.edge_index[1]
            edge_attr = self.edge_weight

            # Use COO format instead of DOK
            A = sp.coo_matrix(
                (edge_attr, (row, col)),
                shape=(self.n_users + self.n_items,
                       self.n_users + self.n_items)
            )

            # Normalize: D^(-1/2) A D^(-1/2)
            sumArr = (A > 0).sum(axis=1)
            rowsum = np.array(sumArr).flatten()
            rowsum[rowsum == 0.0] = 0.0
            d_inv = np.power(rowsum, -0.5)
            d_inv[np.isinf(d_inv)] = 0.0
            d_mat_inv = sp.diags(d_inv)

            norm_adj = d_mat_inv @ A @ d_mat_inv

            # Convert to COO for storage
            return norm_adj.tocoo()

        LightGCN.get_norm_adj_mat = patched_get_norm_adj_mat
        print("✓ LightGCN scipy compatibility patch applied")
    except ImportError:
        print("RecBole not yet imported — patch will be applied when RecBole initializes")


_patch_lightgcn_scipy()

In [5]:
# ── Verify GPU and imports ────────────────────────────────────────
import torch
import numpy as np
import pandas as pd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} — device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected. Training will be slow. "
          "Go to Runtime → Change runtime type → GPU.")

In [6]:
# ── Add project root to sys.path ─────────────────────────────────
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Quick sanity check
from src.recommender.recbole_formatter import format_category_for_recbole
from src.engagement.deepfm import DeepFMWrapper
from src.reranking.carbon_reranker import CarbonReranker
from src.evaluation.metrics import pareto_frontier
print("All modules imported ✓")

In [7]:
# ── Verify data exists ───────────────────────────────────────────
from pathlib import Path

INTERIM_DIR = Path("data/interim")
for split in ["train", "val", "test"]:
    for cat in ["electronics", "home_and_kitchen", "sports_and_outdoors"]:
        p = INTERIM_DIR / split / f"{cat}.csv"
        if p.exists():
            n = sum(1 for _ in open(p)) - 1
            print(f"  ✓ {p}  ({n:,} rows)")
        else:
            print(f"  ✗ {p}  MISSING")

## Configuration

Edit these to control what gets trained:

In [ ]:
# ── Pipeline configuration ───────────────────────────────────────
CATEGORIES = ["electronics"]

# Which models to run — pick one or all four:
# MODELS = ["BPR"]                                   # single model
MODELS = ["BPR", "NeuMF", "SASRec", "LightGCN"]     # all four

# Speed / subset controls
MAX_TRAIN_USERS = 5000        # Subset training data to this many users (None = use all)
N_USERS = 500                 # Users to include in re-ranking evaluation
N_EPOCHS = 2                  # RecBole training epochs (default configs use 50)
BATCH_SIZE = 1024             # Training & eval batch size

TOP_K_CANDIDATES = 100        # Candidates per user from RecBole
TOP_K_RERANK = 10             # Final list length per user

# λ values for carbon-aware re-ranking sweep
LAMBDA_VALUES = [0.0, 0.25, 0.5, 0.75, 0.80, 0.85, 0.9, 0.95, 1.0]

print(f"Categories:       {CATEGORIES}")
print(f"Models:           {MODELS}")
print(f"MAX_TRAIN_USERS:  {MAX_TRAIN_USERS}")
print(f"N_USERS (rerank): {N_USERS}")
print(f"N_EPOCHS:         {N_EPOCHS}")
print(f"Device:           {DEVICE}")
print(f"λ values:         {LAMBDA_VALUES}")

---
## Steps 1–4: Train, Re-rank, Evaluate (per model)

For each model in `MODELS`, this cell runs the full pipeline:
1. Train RecBole and extract top-K relevance scores
2. Use relevance scores as engagement proxy (DeepFM skipped)
3. Carbon-aware re-ranking across all lambda values
4. Evaluate and save plots (trade-off curve, lambda sensitivity)

In [ ]:
import json
import time
import matplotlib.pyplot as plt
from datetime import datetime
from tqdm.notebook import tqdm

from src.recommender.recbole_formatter import format_category_for_recbole, _concat_interim_splits
from src.recommender.trainer import train_and_score
from src.reranking.carbon_reranker import CarbonReranker, build_test_set, compute_reranking_metrics
from src.evaluation.metrics import (
    pareto_frontier,
    build_summary_table,
    plot_tradeoff_curve,
    plot_lambda_sensitivity,
)

Path("output/results").mkdir(parents=True, exist_ok=True)
Path("output/figures").mkdir(parents=True, exist_ok=True)

all_reranking_results = {}
model_timings = {}

run_metadata = {
    "timestamp": datetime.now().isoformat(),
    "models": MODELS,
    "categories": CATEGORIES,
    "max_train_users": MAX_TRAIN_USERS,
    "n_users_rerank": N_USERS,
    "n_epochs": N_EPOCHS,
    "batch_size": BATCH_SIZE,
    "top_k_candidates": TOP_K_CANDIDATES,
    "top_k_rerank": TOP_K_RERANK,
    "lambda_values": LAMBDA_VALUES,
    "device": DEVICE,
}
train_label = f"{MAX_TRAIN_USERS} users" if MAX_TRAIN_USERS else "all users"
subtitle = f"Train={train_label}  Rerank={N_USERS} users  Epochs={N_EPOCHS}  {datetime.now().strftime('%Y-%m-%d %H:%M')}"

for model in tqdm(MODELS, desc="Models", unit="model"):
    t_model_start = time.time()

    for cat in tqdm(CATEGORIES, desc=f"  {model} categories", leave=False, unit="cat"):
        print(f"\n{'='*70}")
        print(f"  [{model}] {cat}")
        print(f"{'='*70}")

        # ── Step 1: Train RecBole ────────────────────────────────
        output_dir, dataset_name = format_category_for_recbole(cat, max_users=MAX_TRAIN_USERS)

        config_file = Path(f"configs/recbole/{model.lower()}.yaml")
        config_file = config_file if config_file.exists() else None

        overrides = {
            "epochs": N_EPOCHS,
            "train_batch_size": BATCH_SIZE,
            "eval_batch_size": BATCH_SIZE,
        }
        if model == "SASRec":
            overrides["train_neg_sample_args"] = None
            overrides["eval_args"] = {
                "split": {"LS": "valid_and_test"},
                "group_by": "user",
                "order": "TO",
                "mode": "full",
            }
            overrides["load_col"] = {"inter": ["user_id", "item_id", "timestamp"]}
    
        if DEVICE == "cuda":
            overrides["device"] = DEVICE
        

        scores_df, eval_results = train_and_score(
            dataset_name=dataset_name,
            model_name=model,
            config_file=config_file,
            overrides=overrides,
            top_k=TOP_K_CANDIDATES,
        )
        print(f"  RecBole eval: {eval_results}")
        print(f"  Scores: {len(scores_df):,} (user, item) pairs")

        # ── Step 2: Use relevance as engagement (DeepFM skipped) ─
        engagement_df = scores_df.rename(columns={"relevance_score": "engagement_score"})

        user_subset = engagement_df["user_id"].unique()[:N_USERS]
        engagement_df = engagement_df[engagement_df["user_id"].isin(user_subset)].copy()
        print(f"  Engagement subset: {len(engagement_df):,} pairs ({N_USERS} users)")

        # ── Step 3: Carbon-aware re-ranking ──────────────────────
        interactions = _concat_interim_splits(cat)
        carbon_df = (
            interactions[["parent_asin", "pcf"]]
            .drop_duplicates(subset="parent_asin")
            .copy()
        )
        test_interactions = build_test_set(interactions)
        test_interactions = test_interactions[test_interactions["user_id"].isin(user_subset)].copy()

        reranker = CarbonReranker(top_k=TOP_K_RERANK)
        all_metrics = []

        for lam in tqdm(LAMBDA_VALUES, desc=f"    {model} λ sweep", leave=False, unit="λ"):
            ranked = reranker.rerank(engagement_df, carbon_df, lam)
            metrics = compute_reranking_metrics(ranked, test_interactions, lam, k=TOP_K_RERANK)
            all_metrics.append(metrics)

            out_path = Path(f"output/results/{cat}_{model}_reranked_{lam:.3f}.parquet")
            ranked.to_parquet(out_path, index=False)

        all_reranking_results[(cat, model)] = all_metrics

        baseline = next(m for m in all_metrics if m["lambda"] == 0.0)
        greenest = min(all_metrics, key=lambda m: m["avg_carbon_kg"])
        summary = {
            "category": cat,
            "model": model,
            "max_train_users": MAX_TRAIN_USERS,
            "n_users_rerank": N_USERS,
            "n_epochs": N_EPOCHS,
            "baseline_carbon_kg": baseline["avg_carbon_kg"],
            "baseline_NDCG@10": baseline["NDCG@10"],
            "greenest_lambda": greenest["lambda"],
            "greenest_carbon_kg": greenest["avg_carbon_kg"],
            "greenest_NDCG@10": greenest["NDCG@10"],
        }
        metrics_path = Path(f"output/results/{cat}_{model}_reranking_metrics.json")
        with open(metrics_path, "w") as f:
            json.dump({"summary": summary, "per_lambda": all_metrics, "run": run_metadata}, f, indent=2)

        print(f"\n  {'λ':>6}  {'NDCG@10':>8}  {'Recall@10':>10}  {'MRR':>6}  {'Avg CO₂ (kg)':>13}")
        print(f"  {'─'*6}  {'─'*8}  {'─'*10}  {'─'*6}  {'─'*13}")
        for m in all_metrics:
            print(f"  {m['lambda']:6.3f}  {m['NDCG@10']:8.4f}  {m['Recall@10']:10.4f}  "
                  f"{m['MRR']:6.4f}  {m['avg_carbon_kg']:13.2f}")

        # ── Step 4: Evaluate + save plots ────────────────────────
        front = pareto_frontier(all_metrics)
        summary_df = build_summary_table(all_metrics)
        summary_df.to_csv(f"output/results/{cat}_{model}_evaluation_summary.csv", index=False)

        pareto_path = Path(f"output/results/{cat}_{model}_pareto.json")
        with open(pareto_path, "w") as f:
            json.dump(front, f, indent=2)

        fig = plot_tradeoff_curve(all_metrics, cat, model, save=True, show=False)
        fig.suptitle(f"Engagement vs Carbon — {cat} ({model})", fontsize=14, fontweight="bold", y=1.02)
        fig.text(0.5, -0.02, subtitle, ha="center", fontsize=9, color="gray")
        fig.savefig(f"output/figures/{cat}_{model}_tradeoff.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        fig = plot_lambda_sensitivity(all_metrics, cat, model, save=True, show=False)
        fig.suptitle(f"λ Sensitivity — {cat} ({model})", fontsize=14, fontweight="bold", y=1.02)
        fig.text(0.5, -0.02, subtitle, ha="center", fontsize=9, color="gray")
        fig.savefig(f"output/figures/{cat}_{model}_lambda_sensitivity.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        display(summary_df)

    elapsed = time.time() - t_model_start
    model_timings[model] = elapsed
    print(f"\n  ⏱ {model} completed in {elapsed:.0f}s ({elapsed/60:.1f} min)")

# Save run config
with open("output/results/run_config.json", "w") as f:
    run_metadata["model_timings_seconds"] = model_timings
    json.dump(run_metadata, f, indent=2)

print(f"\n{'='*70}")
print("All models complete. Timings:")
for m, t in model_timings.items():
    print(f"  {m:10s}  {t:6.0f}s  ({t/60:.1f} min)")
print(f"  {'TOTAL':10s}  {sum(model_timings.values()):6.0f}s  ({sum(model_timings.values())/60:.1f} min)")
print(f"{'='*70}")

---
## Multi-Model Comparison Plots

In [ ]:
# ── Multi-Model Pareto Frontier Comparison ───────────────────────
from src.evaluation.metrics import pareto_frontier
import matplotlib.pyplot as plt
import json

colors = {"BPR": "tab:blue", "NeuMF": "tab:orange", "SASRec": "tab:green", "LightGCN": "tab:red"}

for cat in CATEGORIES:
    fig, ax = plt.subplots(figsize=(10, 6))

    for model in MODELS:
        metrics_path = f"output/results/{cat}_{model}_reranking_metrics.json"
        try:
            with open(metrics_path, "r") as f:
                metrics_data = json.load(f)["per_lambda"]
            front = pareto_frontier(metrics_data)
            x = [pt["avg_carbon_kg"] for pt in front]
            y = [pt["NDCG@10"] for pt in front]
            ax.plot(x, y, marker="o", label=model, color=colors.get(model, "gray"), linewidth=2)
        except Exception as e:
            print(f"Could not load results for {model}: {e}")

    ax.set_xlabel("Average Carbon (kg CO₂e)", fontsize=12)
    ax.set_ylabel("NDCG@10", fontsize=12)
    ax.set_title(f"Pareto Frontier Comparison — {cat}", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.text(0.5, -0.02, subtitle, ha="center", fontsize=9, color="gray")
    fig.tight_layout()

    Path("output/figures").mkdir(parents=True, exist_ok=True)
    fig.savefig(f"output/figures/{cat}_multi_model_pareto.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved: output/figures/{cat}_multi_model_pareto.png")

In [ ]:
# ── Summary: Best Pareto Points (≥10% carbon reduction) ─────────
import pandas as pd

print("=== Best Pareto Points (≥10% carbon reduction) ===\n")
best_points = []
for cat in CATEGORIES:
    for model in MODELS:
        metrics_path = f"output/results/{cat}_{model}_reranking_metrics.json"
        try:
            with open(metrics_path, "r") as f:
                metrics_data = json.load(f)["per_lambda"]
            baseline_carbon = metrics_data[0]["avg_carbon_kg"]
            best = None
            for pt in sorted(metrics_data, key=lambda p: -p["NDCG@10"]):
                reduction = (baseline_carbon - pt["avg_carbon_kg"]) / max(baseline_carbon, 1e-9)
                if reduction >= 0.10:
                    best = pt
                    break
            if best:
                print(f"{cat:20s} | {model:8s} | λ={best['lambda']:.2f} | "
                      f"NDCG@10={best['NDCG@10']:.4f} | "
                      f"carbon={best['avg_carbon_kg']:.2f} kg (−{reduction*100:.1f}%)")
                best_points.append({
                    "category": cat, "model": model,
                    "lambda": best["lambda"],
                    "NDCG@10": best["NDCG@10"],
                    "avg_carbon_kg": best["avg_carbon_kg"],
                })
            else:
                print(f"{cat:20s} | {model:8s} | No point with ≥10% carbon reduction")
        except Exception as e:
            print(f"{cat:20s} | {model:8s} | Results missing: {e}")

if best_points:
    df_best = pd.DataFrame(best_points)
    display(df_best)

In [ ]:
# ── Best Pareto Points Scatter Plot ──────────────────────────────
if best_points:
    fig, ax = plt.subplots(figsize=(10, 6))
    for model in MODELS:
        sub = df_best[df_best["model"] == model]
        ax.scatter(sub["avg_carbon_kg"], sub["NDCG@10"], label=model,
                   s=100, color=colors.get(model, "gray"), zorder=3)
        for _, row in sub.iterrows():
            ax.text(row["avg_carbon_kg"] + 0.01, row["NDCG@10"] + 0.002,
                    row["category"], fontsize=9)

    ax.set_xlabel("Avg Carbon (kg CO₂e)", fontsize=12)
    ax.set_ylabel("NDCG@10", fontsize=12)
    ax.set_title("Best Pareto Points by Model (≥10% carbon reduction)",
                 fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.text(0.5, -0.02, subtitle, ha="center", fontsize=9, color="gray")
    fig.tight_layout()
    fig.savefig("output/figures/best_pareto_points_by_model.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved: output/figures/best_pareto_points_by_model.png")
else:
    print("No best points found — skipping scatter plot.")

In [ ]:
# ── Timing bar chart ─────────────────────────────────────────────
if model_timings:
    fig, ax = plt.subplots(figsize=(8, 4))
    models_sorted = list(model_timings.keys())
    times_min = [model_timings[m] / 60 for m in models_sorted]
    bars = ax.barh(models_sorted, times_min, color=[colors.get(m, "gray") for m in models_sorted])
    ax.set_xlabel("Time (minutes)", fontsize=12)
    ax.set_title("Training + Evaluation Time per Model", fontsize=14, fontweight="bold")
    for bar, t in zip(bars, times_min):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f"{t:.1f} min", va="center", fontsize=10)
    fig.tight_layout()
    fig.savefig("output/figures/model_timings.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Saved: output/figures/model_timings.png")

---
## Download Results

Run the cell below to zip all outputs and download them.

In [ ]:
# ── Zip and download results ─────────────────────────────────────
import shutil
from pathlib import Path
from google.colab import files

output_dir = Path("output")
if not output_dir.exists():
    print("⚠️ No output directory found")
else:
    # Create zip file
    shutil.make_archive("carbon_aware_results", "zip", ".", "output")
    zip_path = Path("carbon_aware_results.zip")
    zip_size_mb = zip_path.stat().st_size / (1024**2)

    print(f"✓ Created: carbon_aware_results.zip ({zip_size_mb:.1f} MB)")
    print("\nFiles included:")
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            size_kb = p.stat().st_size / 1024
            print(f"  {p}  ({size_kb:.0f} KB)")

    print(f"\nDownloading...")
    files.download("carbon_aware_results.zip")

---
## Output Summary

| Output | Location |
|--------|----------|
| RecBole scores | `output/results/<cat>_<model>_scores.parquet` |
| Re-ranked lists | `output/results/<cat>_<model>_reranked_<lambda>.parquet` |
| Metrics JSON | `output/results/<cat>_<model>_reranking_metrics.json` |
| Summary CSV | `output/results/<cat>_<model>_evaluation_summary.csv` |
| Pareto JSON | `output/results/<cat>_<model>_pareto.json` |
| Run config | `output/results/run_config.json` |
| Trade-off plot | `output/figures/<cat>_<model>_tradeoff.png` |
| Lambda sensitivity | `output/figures/<cat>_<model>_lambda_sensitivity.png` |
| Multi-model Pareto | `output/figures/<cat>_multi_model_pareto.png` |
| Best points scatter | `output/figures/best_pareto_points_by_model.png` |
| Timing chart | `output/figures/model_timings.png` |